In [ ]:
import os, sys

print(os.path.abspath(os.path.join(os.getcwd(), '.')))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from extern.network_backend.g2.network_v5 import Network

import time
import random
import pandas as pd
import numpy as np
import ast
from collections import defaultdict
import g2_lib as g2

# Helper function from network_v5.py to create JSON objects for the patata library
def string_to_elements(s):
    # Step 1: turn the string into a Python list
    outer_list = ast.literal_eval(s)

    # Step 2: build the list of dicts
    result = []
    for item in outer_list:
        parts, number = item.split("#")  # split at "#"
        values = ast.literal_eval(parts)  # turn the string list into real list
        result.append({
            "id": item,
            "path": values,
            "num_flows": int(number)
        })
    return result

# --- Experiment Configuration ---
npus_count = [64]
topology = ['ring']
bandwidth = [900] * len(npus_count)
num_npus = np.prod(npus_count)

# Define scenarios
initial_flows_options = [4, 8, 16, 32, 64]
flows_to_change_options = [1, 2, 4, 8, 16, 32]
iterations_per_scenario = 50

results = []

print("--- Starting Network Update Performance Comparison ---")

for initial_flows in initial_flows_options:
    for flows_to_change in flows_to_change_options:
        # Skip scenarios where we try to remove more flows than exist
        if flows_to_change >= initial_flows:
            continue

        flows_to_add = flows_to_change
        flows_to_remove = flows_to_change

        print(f"\nTesting Scenario: Initial={initial_flows}, Add={flows_to_add}, Remove={flows_to_remove}, Iterations={iterations_per_scenario}")

        # Accumulators for timing across iterations
        total_time_add_incremental = 0
        total_time_remove_incremental = 0
        total_time_add_json = 0
        total_time_remove_json = 0

        for _ in range(iterations_per_scenario):
            # --- Setup a base network with initial flows for each iteration ---
            base_net = Network(
                npus_count_per_dim=npus_count,
                bandwidth_per_dim=bandwidth,
                topologies_per_dim=topology,
            )
            
            # Add initial flows
            initial_routes = []
            for i in range(initial_flows):
                src, dest = random.sample(range(num_npus), 2)
                route_path = base_net.get_route(src, dest)
                route_links = [f"{u}-{v}" for u, v in zip(route_path, route_path[1:])]
                initial_routes.append(str(route_links))

            # Create networks for incremental tests and populate them
            net_incremental_add = Network(npus_count_per_dim=npus_count, bandwidth_per_dim=bandwidth, topologies_per_dim=topology)
            net_incremental_remove = Network(npus_count_per_dim=npus_count, bandwidth_per_dim=bandwidth, topologies_per_dim=topology)
            
            for route_str in initial_routes:
                net_incremental_add.add_flowgroup(route_str)
                net_incremental_remove.add_flowgroup(route_str)

            # --- Method 1: Incremental Update (using add_flowgroup/remove_flowgroup) ---

            # Generate routes to add
            routes_to_add_list = []
            for i in range(flows_to_add):
                src, dest = random.sample(range(num_npus), 2)
                route_path = base_net.get_route(src, dest)
                route_links = [f"{u}-{v}" for u, v in zip(route_path, route_path[1:])]
                routes_to_add_list.append(str(route_links))

            # Time the addition of new routes using add_flowgroup
            start_add_inc = time.perf_counter()
            for route_str in routes_to_add_list:
                net_incremental_add.add_flowgroup(route_str)
            total_time_add_incremental += time.perf_counter() - start_add_inc

            # Identify routes to remove from the initial set of flows
            routes_to_remove_list = list(net_incremental_remove.network.flow_names())[:flows_to_remove]

            # Time the removal of routes using remove_flowgroup
            start_remove_inc = time.perf_counter()
            for route_str in routes_to_remove_list:
                net_incremental_remove.remove_flowgroup(route_str)
            total_time_remove_incremental += time.perf_counter() - start_remove_inc

            # --- Method 2: Recreation from JSON ---
            
            # Get the state of the base network
            network_links_json = base_net.network_links_json

            # 2. Combine initial and new routes for ADDITION test
            route_counts_add = defaultdict(int)
            for route_str in initial_routes:
                route_counts_add[route_str] += 1
            for route_str in routes_to_add_list:
                route_counts_add[route_str] += 1
            
            final_flow_names_add = [f"{route_str}#{count}" for route_str, count in route_counts_add.items()]
            routes_in_use_key_add = str(sorted(final_flow_names_add))

            # 3. Time the recreation for ADDITION
            start_add_json = time.perf_counter()
            routes_json_add = string_to_elements(routes_in_use_key_add)
            recreated_net_add = g2.network_from_json_objs(network_links_json, routes_json_add)
            total_time_add_json += time.perf_counter() - start_add_json

            # Time the removal by recreating the network without the removed flows
            # 1. "Remove" the first `flows_to_remove` from our counts
            route_counts_remove = defaultdict(int)
            for route_str in initial_routes:
                route_counts_remove[route_str] += 1

            for route_to_remove in routes_to_remove_list:
                if route_to_remove in route_counts_remove:
                    route_counts_remove[route_to_remove] -= 1
                    if route_counts_remove[route_to_remove] == 0:
                        del route_counts_remove[route_to_remove]

            final_flow_names_remove = [f"{route_str}#{count}" for route_str, count in route_counts_remove.items()]
            routes_in_use_key_remove = str(sorted(final_flow_names_remove))

            # 2. Time the recreation for REMOVAL
            start_remove_json = time.perf_counter()
            routes_json_remove = string_to_elements(routes_in_use_key_remove)
            recreated_net_remove = g2.network_from_json_objs(network_links_json, routes_json_remove)
            total_time_remove_json += time.perf_counter() - start_remove_json
            
            del base_net
            del net_incremental_add
            del net_incremental_remove

        # --- Store Average Results for the Scenario ---
        results.append({
            'initial_flows': initial_flows,
            'flows_to_add': flows_to_add,
            'flows_to_remove': flows_to_remove,
            'time_add_incremental': total_time_add_incremental / iterations_per_scenario,
            'time_add_json': total_time_add_json / iterations_per_scenario,
            'time_remove_incremental': total_time_remove_incremental / iterations_per_scenario,
            'time_remove_json': total_time_remove_json / iterations_per_scenario,
        })

# --- Display Results ---
df_comparison = pd.DataFrame(results)
print("\n--- Performance Comparison Summary (average times in seconds) ---")
display(df_comparison)